# Authorship Verification — Siamese Network + Impostor Projections

**Final Submission — Phase B**

A deep-learning pipeline that verifies whether two texts share the same
author. A BERT-based Siamese network (CNN-BiLSTM head) is trained with
contrastive loss; per-chunk similarity is aggregated into a style signal,
aligned via Dynamic Time Warping, and scored with Isolation Forest
anomaly detection to reach a final genuine/impostor verdict.

**Pipeline**
```
Text A, Text B
  -> BERT (partially frozen) -> per-chunk embeddings
  -> CNN-BiLSTM -> FC -> style embedding
  -> Euclidean distance -> Contrastive Loss            (training)
  -> per-chunk similarity matrix -> DTW-aligned signal
  -> Isolation Forest anomaly score -> tuned threshold -> verdict
```

**Notebook structure**

| Category | Sections |
|---|---|
| A. Setup | 1–3 |
| B. Data | 4 |
| C. Model & Training | 5–8 |
| D. Inference Pipeline | 9–10 |
| E. Evaluation | 11–13 |
| F. Accuracy Improvement | 14 |
| G. Save Results | 15 |


# Category A — Setup

## 1. Preparations

Mount Drive, unzip project code, install dependencies.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip project code from Drive into the local runtime.
import shutil
import os
import zipfile

DRIVE_PROJECT = "/content/drive/MyDrive/Authorship Verification"
ZIP_FILE_PATH = os.path.join(DRIVE_PROJECT, "authorship_verification.zip")
TARGET_DIR = "/content/authorship_verification"

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

if os.path.exists(ZIP_FILE_PATH):
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print(f"Project unzipped to {TARGET_DIR}")
else:
    print(f"Error: zip file not found at {ZIP_FILE_PATH}")

In [ ]:
# Install dependencies with pinned, compatible versions.
# requirements.txt has "numpy>=1.24.0" with no upper bound, which would
# upgrade to numpy 2.x and break scikit-learn-extra's binary compatibility.
# We install pinned versions instead and skip requirements.txt entirely.

!pip uninstall -y numpy scipy scikit-learn scikit-learn-extra -q
!pip install numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2 -q
!pip install --force-reinstall --no-deps scikit-learn-extra -q
!pip install fastdtw -q
!pip install torch transformers pandas matplotlib tqdm datasets -q

print("Dependencies installed: numpy==1.26.4, scipy==1.11.4, scikit-learn==1.3.2")

In [ ]:
# Required: restart the runtime now.
# numpy may already be loaded in memory from Colab's own startup imports,
# so the newly installed version is not visible until the runtime restarts.
print("Go to: Runtime -> Restart session")
print("Then continue from the NEXT cell (do not re-run the cells above).")

In [ ]:
# Sanity check on the project structure.
# Runs in a separate subprocess, so it is unaffected by the numpy issue above.
!cd /content/authorship_verification && python verify.py

---
### Stop here — restart the runtime

**Runtime -> Restart session**, then continue from the next cell.
The installed packages remain on disk; no need to repeat Section 1.
---

## 2. Patches — Fixed Files

Before training, several source files are rewritten with fixes found
during development. Each row maps to a concrete issue observed when
comparing a prior run's output against the original source code.

| File | Fix | Reason |
|---|---|---|
| `config.py` | `num_epochs=10`, `early_stopping_patience=2` | Prevents overfitting (best epoch was reached well before training stopped in a prior run) |
| `config.py` | `freeze_bert_layers=6` (was 8) | Unfreezes 2 more BERT layers for finer-grained learning |
| `config.py` | `contrastive_margin=1.5` (was 1.0) | Clearer separation between genuine/impostor embeddings |
| `config.py` | `isolation_forest_contamination=0.45` | Matches the dataset's actual ~50/50 genuine/impostor split |
| `pipeline/anomaly.py` | Median-threshold split, not biased K-Medoids | Fixed a strong bias toward predicting "genuine" |
| `pipeline/anomaly.py` | 4 extra features (correlation, ranges, DTW/embedding ratio) | More signal for the Isolation Forest |
| `pipeline/signal.py` | `aggregation="combined"` | Richer style signal than any single aggregation method |
| `pipeline/dtw.py` | `fastdtw` + z-score signal normalization | Faster, scale-fair signal comparison |
| `data/preprocessing.py` | Cap chunks at 6 per text | Prevents CUDA out-of-memory on 16GB GPUs |
| `config.py` | `batch_size=4` set as the file's own default (not overridden later) | Found in testing: setting `batch_size` AFTER `create_dataloaders_from_splits()` already ran has no effect on the already-built loaders and causes a CUDA OOM crash mid-training |
| `training/trainer.py` | One unified "best loss" tracker | Checkpoint-saving and early-stopping previously used two independent trackers that could disagree |


In [ ]:
# Add the project directory to Python's import path.
# If the runtime was restarted, sys.path resets — re-run this cell first.
import sys
import os
sys.path.insert(0, "/content/authorship_verification")

In [ ]:
# config.py — all hyperparameters in one place
config_code = """\"\"\"
Central configuration for the Authorship Verification Framework.
\"\"\"
import os
from dataclasses import dataclass, field
from typing import Optional
import torch

@dataclass
class PathConfig:
    root_dir: str = "."
    data_dir: str = "data_raw"
    checkpoint_dir: str = "checkpoints"
    results_dir: str = "results"
    logs_dir: str = "logs"
    def __post_init__(self):
        for d in [self.data_dir, self.checkpoint_dir, self.results_dir, self.logs_dir]:
            os.makedirs(os.path.join(self.root_dir, d), exist_ok=True)

@dataclass
class DataConfig:
    bert_model_name: str = "bert-base-uncased"
    max_chunk_tokens: int = 512
    min_chunks_per_text: int = 1
    impostor_ratio: float = 1.0
    num_impostors_per_author: int = 5
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    dataset_source: str = "pan2020"

@dataclass
class ModelConfig:
    bert_model_name: str = "bert-base-uncased"
    bert_hidden_size: int = 768
    freeze_bert_layers: int = 6
    cnn_num_filters: int = 128
    cnn_kernel_sizes: tuple = (3, 4, 5)
    cnn_dropout: float = 0.3
    bilstm_hidden_size: int = 128
    bilstm_num_layers: int = 2
    bilstm_dropout: float = 0.3
    embedding_dim: int = 256
    fc_hidden_dim: int = 128
    fc_dropout: float = 0.4

@dataclass
class TrainingConfig:
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    adam_epsilon: float = 1e-8
    warmup_steps: int = 500
    scheduler_type: str = "linear"
    batch_size: int = 4         # 8/16 caused CUDA OOM on T4 with multi-chunk texts
    num_epochs: int = 10
    gradient_accumulation_steps: int = 8  # effective batch = 4 * 8 = 32
    early_stopping_patience: int = 2
    early_stopping_min_delta: float = 0.005
    save_every_n_epochs: int = 2
    keep_top_k_checkpoints: int = 3
    contrastive_margin: float = 1.5
    seed: int = 42

@dataclass
class PipelineConfig:
    dtw_window_size: Optional[int] = None
    isolation_forest_contamination: float = 0.45
    isolation_forest_n_estimators: int = 200
    isolation_forest_random_state: int = 42
    kmedoids_n_clusters: int = 2
    kmedoids_random_state: int = 42

@dataclass
class Config:
    paths: PathConfig = field(default_factory=PathConfig)
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    pipeline: PipelineConfig = field(default_factory=PipelineConfig)

    @property
    def device(self) -> torch.device:
        if torch.cuda.is_available():
            return torch.device("cuda")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")

cfg = Config()
"""

with open("/content/authorship_verification/config.py", "w") as f:
    f.write(config_code)

print("config.py updated.")

In [ ]:
# pipeline/anomaly.py — balanced threshold split + extended features
anomaly_code = """\"\"\"
Anomaly Detection Module
========================
Isolation Forest + threshold clustering for the final verdict.
\"\"\"
import numpy as np
from typing import Dict
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from config import cfg


class AuthorshipAnomalyDetector:
    \"\"\"Final decision: Isolation Forest -> threshold clustering -> verdict.\"\"\"

    def __init__(self, config=None):
        c = config or cfg.pipeline
        self.isolation_forest = IsolationForest(
            contamination=c.isolation_forest_contamination,
            n_estimators=c.isolation_forest_n_estimators,
            random_state=c.isolation_forest_random_state,
            max_features=1.0,
        )
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit(self, genuine_features: np.ndarray):
        \"\"\"Fit on GENUINE pairs only (learn what \'normal\' looks like).\"\"\"
        features_scaled = self.scaler.fit_transform(genuine_features)
        self.isolation_forest.fit(features_scaled)
        self.is_fitted = True

    def predict(self, features: np.ndarray) -> Dict[str, np.ndarray]:
        if not self.is_fitted:
            raise RuntimeError("Call fit() first")
        features_scaled = self.scaler.transform(features)
        predictions = self.isolation_forest.predict(features_scaled)
        anomaly_scores = self.isolation_forest.decision_function(features_scaled)
        return {
            "predictions": predictions,
            "anomaly_scores": anomaly_scores,
            "is_genuine": predictions == 1,
        }

    def predict_single(self, features: np.ndarray) -> Dict[str, float]:
        result = self.predict(features.reshape(1, -1))
        score = result["anomaly_scores"][0]
        is_genuine = result["is_genuine"][0]
        confidence = 1.0 / (1.0 + np.exp(-score * 2))
        return {
            "verdict": "genuine" if is_genuine else "impostor",
            "anomaly_score": float(score),
            "confidence": float(confidence if is_genuine else 1 - confidence),
        }


def extract_pair_features(dtw_distance, similarity_signal_a,
                           similarity_signal_b=None,
                           mean_embedding_distance=None) -> np.ndarray:
    \"\"\"Build a 16-dim feature vector for one text pair.\"\"\"
    features = [dtw_distance]

    if len(similarity_signal_a) > 0:
        features.extend([
            np.mean(similarity_signal_a), np.std(similarity_signal_a),
            np.min(similarity_signal_a),  np.max(similarity_signal_a),
            np.median(similarity_signal_a), np.var(similarity_signal_a),
        ])
    else:
        features.extend([0.0] * 6)

    if similarity_signal_b is not None and len(similarity_signal_b) > 0:
        features.extend([
            np.mean(similarity_signal_b), np.std(similarity_signal_b),
            np.min(similarity_signal_b),  np.max(similarity_signal_b),
        ])
    else:
        features.extend([0.0] * 4)

    emb = mean_embedding_distance if mean_embedding_distance is not None else 0.0
    features.append(emb)

    if similarity_signal_b is not None and len(similarity_signal_b) > 0 and len(similarity_signal_a) > 0:
        min_len = min(len(similarity_signal_a), len(similarity_signal_b))
        corr = np.corrcoef(similarity_signal_a[:min_len], similarity_signal_b[:min_len])[0, 1]
        features.append(0.0 if np.isnan(corr) else corr)
        features.append(np.max(similarity_signal_a) - np.min(similarity_signal_a))
        features.append(np.max(similarity_signal_b) - np.min(similarity_signal_b))
    else:
        features.extend([0.0, 0.0, 0.0])

    features.append(dtw_distance / (emb + 1e-8))

    return np.array(features, dtype=np.float64)


def kmedoids_cluster(anomaly_scores: np.ndarray, n_clusters: int = 2) -> np.ndarray:
    \"\"\"
    Split anomaly scores into genuine/impostor via a median threshold.
    Label 1 = genuine (higher anomaly score = more "normal").
    \"\"\"
    threshold = np.median(anomaly_scores)
    labels = (anomaly_scores >= threshold).astype(int)
    means = [anomaly_scores[labels == 0].mean(), anomaly_scores[labels == 1].mean()]
    if means[0] > means[1]:
        labels = 1 - labels
    return labels
"""

with open("/content/authorship_verification/pipeline/anomaly.py", "w") as f:
    f.write(anomaly_code)

print("pipeline/anomaly.py updated.")

In [ ]:
# pipeline/signal.py — richer "combined" aggregation
signal_code = """\"\"\"
Signal Construction Module
==========================
Converts chunk-level similarity scores into a 1D style signal.
\"\"\"
from typing import List
import numpy as np
import torch


def build_similarity_signal(similarity_matrix, aggregation: str = "combined") -> np.ndarray:
    \"\"\"
    Aggregation options: diagonal, row_max, row_mean, col_max, combined.
    "combined" blends row_max + row_mean + col_max + col_mean for a
    richer signal than any single method alone.
    \"\"\"
    mat = similarity_matrix.numpy() if isinstance(similarity_matrix, torch.Tensor) else similarity_matrix
    n_a, n_b = mat.shape

    if aggregation == "diagonal":
        diag_len = min(n_a, n_b)
        return np.array([mat[i, i] for i in range(diag_len)])
    elif aggregation == "row_max":
        return np.max(mat, axis=1)
    elif aggregation == "row_mean":
        return np.mean(mat, axis=1)
    elif aggregation == "col_max":
        return np.max(mat, axis=0)
    elif aggregation == "combined":
        row_max = np.max(mat, axis=1)
        row_mean = np.mean(mat, axis=1)
        col_max = np.max(mat, axis=0)
        col_mean = np.mean(mat, axis=0)
        min_len = min(len(row_max), len(col_max))
        return (row_max[:min_len]  * 0.4 +
                row_mean[:min_len] * 0.3 +
                col_max[:min_len]  * 0.2 +
                col_mean[:min_len] * 0.1)
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")


def build_style_signal(chunk_embeddings, chunk_mask, reference_embedding=None) -> np.ndarray:
    \"\"\"Per-chunk cosine similarity to the text\'s mean (or given) embedding.\"\"\"
    valid = chunk_mask.bool()
    embeddings = chunk_embeddings[valid].detach().cpu()
    if len(embeddings) == 0:
        return np.array([])
    if reference_embedding is None:
        reference = embeddings.mean(dim=0, keepdim=True)
    else:
        reference = reference_embedding.unsqueeze(0).detach().cpu()
    similarities = torch.nn.functional.cosine_similarity(
        embeddings, reference.expand_as(embeddings), dim=1
    )
    return similarities.numpy()


def aggregate_batch_signals(similarity_matrices: List, aggregation: str = "combined") -> List[np.ndarray]:
    return [build_similarity_signal(mat, aggregation) for mat in similarity_matrices]
"""

with open("/content/authorship_verification/pipeline/signal.py", "w") as f:
    f.write(signal_code)

print("pipeline/signal.py updated.")

In [ ]:
# pipeline/dtw.py — fastdtw + signal normalization
dtw_code = """\"\"\"
Dynamic Time Warping (DTW) Module
==================================
Aligns two style signals of possibly different lengths.
\"\"\"
import numpy as np
from typing import Tuple

try:
    from fastdtw import fastdtw
    from scipy.spatial.distance import euclidean
    _FASTDTW = True
except ImportError:
    _FASTDTW = False


def compute_dtw_distance(signal_a: np.ndarray, signal_b: np.ndarray, window: int = None) -> float:
    \"\"\"
    DTW distance between two 1D signals. Signals are z-score normalized
    first so comparisons are fair across different scales. Uses fastdtw
    when available (much faster than the pure-Python fallback).
    \"\"\"
    if len(signal_a) == 0 or len(signal_b) == 0:
        return float("inf")

    def normalize(s):
        std = np.std(s)
        return (s - np.mean(s)) / (std if std > 1e-8 else 1.0)

    a = normalize(signal_a)
    b = normalize(signal_b)

    if _FASTDTW:
        dist, _ = fastdtw(a.reshape(-1, 1), b.reshape(-1, 1), dist=euclidean)
        return float(dist)

    n, m = len(a), len(b)
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0
    for i in range(1, n + 1):
        j_start = max(1, i - window) if window else 1
        j_end   = min(m + 1, i + window + 1) if window else m + 1
        for j in range(j_start, j_end):
            cost = abs(a[i-1] - b[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1])
    return float(dtw[n, m])


def compute_dtw_with_path(signal_a, signal_b, window=None):
    \"\"\"DTW distance AND the optimal alignment path (for plotting).\"\"\"
    n, m = len(signal_a), len(signal_b)
    if n == 0 or m == 0:
        return float("inf"), []
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0
    for i in range(1, n + 1):
        j_start = max(1, i - window) if window else 1
        j_end   = min(m + 1, i + window + 1) if window else m + 1
        for j in range(j_start, j_end):
            cost = abs(signal_a[i-1] - signal_b[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1])
    path, i, j = [], n, m
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        if i == 0: j -= 1
        elif j == 0: i -= 1
        else:
            argmin = np.argmin([dtw[i-1, j-1], dtw[i-1, j], dtw[i, j-1]])
            if argmin == 0: i -= 1; j -= 1
            elif argmin == 1: i -= 1
            else: j -= 1
    path.reverse()
    return float(dtw[n, m]), path
"""

with open("/content/authorship_verification/pipeline/dtw.py", "w") as f:
    f.write(dtw_code)

print("pipeline/dtw.py updated.")

In [ ]:
# data/preprocessing.py — cap chunks per text (CUDA OOM fix)
preprocessing_code = """\"\"\"
Text Preprocessing Module
=========================
Cleans raw text, tokenizes with BERT, and splits into fixed-size chunks.
\"\"\"
import re
from typing import Dict, List
import torch
from transformers import BertTokenizer
from config import cfg


class TextPreprocessor:
    \"\"\"Cleans raw text and prepares it for BERT tokenization.\"\"\"

    def __init__(self):
        self.tokenizer = BertTokenizer.from_pretrained(cfg.data.bert_model_name)

    def clean(self, text: str) -> str:
        \"\"\"Minimal cleaning that preserves stylistic features.\"\"\"
        text = re.sub(r"http\\S+|www\\.\\S+", "", text)
        text = re.sub(r"\\S+@\\S+\\.\\S+", "", text)
        text = re.sub(r"[^\\x20-\\x7E\\n\\t]", "", text)
        text = re.sub(r"[ \\t]+", " ", text)
        text = re.sub(r"\\n{3,}", "\\n\\n", text)
        return text.strip()

    def chunk_text(self, text: str, max_chunks: int = 6) -> List[Dict]:
        \"\"\"
        Split text into chunks of at most max_chunk_tokens tokens.
        max_chunks caps the number of chunks per text (OOM fix): without
        it, long texts produce many chunks, and batch_size * num_chunks
        * 2 towers can exceed a 16GB GPU\'s memory during the BERT forward pass.
        \"\"\"
        max_len = cfg.data.max_chunk_tokens
        all_tokens = self.tokenizer.encode(text, add_special_tokens=False)
        if len(all_tokens) == 0:
            return []

        window_size = max_len - 2
        max_total_tokens = window_size * max_chunks
        all_tokens = all_tokens[:max_total_tokens]

        chunks = []
        for start in range(0, len(all_tokens), window_size):
            window = all_tokens[start : start + window_size]
            input_ids = (
                [self.tokenizer.cls_token_id]
                + window
                + [self.tokenizer.sep_token_id]
            )
            attention_mask = [1] * len(input_ids)
            padding_length = max_len - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length

            chunks.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "token_type_ids": torch.zeros(max_len, dtype=torch.long),
            })

            if len(chunks) >= max_chunks:
                break

        return chunks

    def preprocess(self, text: str) -> List[Dict]:
        \"\"\"Full pipeline: clean -> chunk.\"\"\"
        cleaned = self.clean(text)
        if not cleaned:
            return []
        return self.chunk_text(cleaned)


def validate_chunks(chunks: List[Dict]) -> bool:
    \"\"\"Validation helper for test TP-02.\"\"\"
    max_len = cfg.data.max_chunk_tokens
    tokenizer = BertTokenizer.from_pretrained(cfg.data.bert_model_name)
    for i, chunk in enumerate(chunks):
        ids = chunk["input_ids"]
        if len(ids) != max_len:
            print(f"Chunk {i}: expected length {max_len}, got {len(ids)}")
            return False
        if ids[0].item() != tokenizer.cls_token_id:
            print(f"Chunk {i}: missing [CLS] token at position 0")
            return False
    return True
"""

with open("/content/authorship_verification/data/preprocessing.py", "w") as f:
    f.write(preprocessing_code)

print("data/preprocessing.py updated — chunks capped at 6 per text.")

In [ ]:
# training/trainer.py — unify the two "best loss" trackers
# The original had two independent best-loss trackers: train()'s local
# variable (strict "<") and EarlyStopping's own tracker ("> best - min_delta").
# These could disagree on a noisy loss curve. This patch makes the checkpoint
# save use the exact same condition as EarlyStopping.
trainer_path = "/content/authorship_verification/training/trainer.py"
with open(trainer_path, "r") as f:
    trainer_src = f.read()

old_block = """        best_val_loss = float(\'inf\')

        for epoch in range(cfg.training.num_epochs):"""
new_block = """        # Single source of truth for "best": self.early_stopping.best_loss
        for epoch in range(cfg.training.num_epochs):"""

if old_block in trainer_src:
    trainer_src = trainer_src.replace(old_block, new_block)
else:
    print("NOTE: expected best_val_loss block not found — skipping this replacement.")

old_save_check = """            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self._save_checkpoint(epoch, val_loss, is_best=True)"""
new_save_check = """            # Checkpoint "best" now matches EarlyStopping's own improvement rule
            es = self.early_stopping
            is_new_best = (
                es.best_loss is None
                or val_loss <= es.best_loss - es.min_delta
            )
            if is_new_best:
                self._save_checkpoint(epoch, val_loss, is_best=True)"""

if old_save_check in trainer_src:
    trainer_src = trainer_src.replace(old_save_check, new_save_check)
else:
    print("NOTE: expected checkpoint-save block not found — skipping this replacement.")

with open(trainer_path, "w") as f:
    f.write(trainer_src)

print("training/trainer.py updated — unified best-loss tracking.")

## 3. Environment Setup

Seed for reproducibility, GPU check.

In [ ]:
import torch
import numpy as np
import random

from config import cfg

random.seed(cfg.training.seed)
np.random.seed(cfg.training.seed)
torch.manual_seed(cfg.training.seed)

print(f"Device: {cfg.device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Category B — Data

## 4. Data Loading

Loads train/val/test splits from PAN20 CSVs using chunked reading
(safe for multi-GB files), then builds DataLoaders.

In [ ]:
from data.dataset import load_pan20_csv

DATA_DIR = "/content/drive/MyDrive/Authorship Verification/data"

print("=" * 60)
print("Loading training data...")
print("=" * 60)
train_pairs = load_pan20_csv(
    os.path.join(DATA_DIR, "pan20_train.csv"),
    max_pairs=20000,
    max_text_length=5000,
    seed=45,
)

print("\n" + "=" * 60)
print("Loading validation data...")
print("=" * 60)
val_pairs = load_pan20_csv(
    os.path.join(DATA_DIR, "pan20_val.csv"),
    max_pairs=2000,
    max_text_length=5000,
    seed=45,
)

print("\n" + "=" * 60)
print("Loading test data...")
print("=" * 60)
test_pairs = load_pan20_csv(
    os.path.join(DATA_DIR, "pan20_test.csv"),
    max_pairs=2000,
    max_text_length=5000,
    seed=45,
)

print(f"\nTotal: {len(train_pairs)} train / {len(val_pairs)} val / {len(test_pairs)} test")

# Report any shortfall vs. the requested max_pairs (the loader can stop
# short without explaining why — usually caused by skipped empty/invalid rows).
for name, pairs, target in [("train", train_pairs, 20000), ("val", val_pairs, 2000), ("test", test_pairs, 2000)]:
    shortfall = target - len(pairs)
    if shortfall > 0:
        pct = shortfall / target * 100
        print(f"NOTE: {name}_pairs got {len(pairs)}/{target} requested ({pct:.1f}% short).")

In [ ]:
# Build DataLoaders. Free train/val raw text from memory, but keep
# val_pairs and test_pairs (small, ~2000 pairs each) — val_pairs is needed
# later for threshold tuning (Section 14), test_pairs for DTW plots (Section 13).
from data.dataset import create_dataloaders_from_splits
import gc

train_loader, val_loader, test_loader = create_dataloaders_from_splits(
    train_pairs, val_pairs, test_pairs
)

del train_pairs
gc.collect()

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Category C — Model and Training

## 5. Model Construction

Build the Siamese network and contrastive loss with hard-example mining.

In [ ]:
from models.siamese import SiameseNetwork
from models.losses import ContrastiveLossWithMining

model = SiameseNetwork()
loss_fn = ContrastiveLossWithMining()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

## 6. Training Configuration

Checkpoint paths and training hyperparameters.

In [ ]:
# Save checkpoints to Drive (survives runtime crashes)
cfg.paths.root_dir = "/content/drive/MyDrive/Authorship Verification"
cfg.paths.checkpoint_dir = "checkpoints"
cfg.paths.results_dir = "results"

cfg.training.num_epochs = 10

# NOTE: batch_size=4 and gradient_accumulation_steps=8 are already set as
# defaults inside config.py (Section 2), BEFORE the DataLoaders were built
# in Section 4. This avoids a real bug found during testing: changing
# batch_size here, AFTER create_dataloaders_from_splits() already ran,
# has no effect on the existing loaders and causes a CUDA OOM crash during
# training. The values below are just displayed for confirmation.

os.makedirs(os.path.join(cfg.paths.root_dir, cfg.paths.checkpoint_dir), exist_ok=True)
os.makedirs(os.path.join(cfg.paths.root_dir, cfg.paths.results_dir), exist_ok=True)

print("Training config:")
print(f"  Epochs:             {cfg.training.num_epochs}")
print(f"  Batch size:         {cfg.training.batch_size}")
print(f"  Grad accumulation:  {cfg.training.gradient_accumulation_steps}")
print(f"  Effective batch:    {cfg.training.batch_size * cfg.training.gradient_accumulation_steps}")
print(f"  Learning rate:      {cfg.training.learning_rate}")
print(f"  Early stopping:     patience={cfg.training.early_stopping_patience}")
print(f"  Contrastive margin: {cfg.training.contrastive_margin}")
print(f"  Checkpoints:        {os.path.join(cfg.paths.root_dir, cfg.paths.checkpoint_dir)}")

In [ ]:
from training.trainer import Trainer

trainer = Trainer(model, train_loader, val_loader, loss_fn)

print("Gradient flow check:")
grad_results = trainer.verify_gradient_flow()
for layer, has_grad in grad_results.items():
    status = "OK" if has_grad else "PROBLEM"
    print(f"  {layer}: {status}")

all_ok = all(grad_results.values())
print(f"\n{'All layers receiving gradients — ready to train!' if all_ok else 'WARNING: some layers have no gradients!'}")

## 7. Training

With `patience=2`, the run should stop early once validation loss
plateaus, saving the best checkpoint to `checkpoints/best_model.pt`.

In [ ]:
history = trainer.train()

## 8. Training Diagnostics

Loss curves and learning-rate schedule.

In [ ]:
from utils.evaluation import plot_training_history

save_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "loss_curves.png")
plot_training_history(history, save_path=save_path)
print(f"Saved to: {save_path}")

# Category D — Inference Pipeline

## 9. Full Inference Pipeline: Siamese -> Signal -> DTW -> Features

Loads the best checkpoint and runs the full pipeline on the test set:
embeddings, per-chunk similarity, style signal, DTW distance, feature
vector. This builds `all_features` / `all_labels`, used by every
section from here on.

In [ ]:
from training.trainer import load_checkpoint

best_path = os.path.join(cfg.paths.root_dir, cfg.paths.checkpoint_dir, "best_model.pt")
load_checkpoint(model, best_path, cfg.device)
print(f"Loaded best checkpoint from: {best_path}")

In [ ]:
from pipeline.signal import build_similarity_signal
from pipeline.dtw import compute_dtw_distance
from pipeline.anomaly import extract_pair_features

model.eval()
model.to(cfg.device)

print("Running inference on test set...")
print("=" * 60)

all_features = []
all_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        outputs = model(batch)

        sim_matrices = model.get_similarity_scores(
            outputs["chunk_embeddings_a"], outputs["chunk_embeddings_b"],
            batch["chunk_mask_a"], batch["chunk_mask_b"],
        )

        for i in range(len(sim_matrices)):
            signal_a = build_similarity_signal(sim_matrices[i], aggregation="combined")
            signal_b = build_similarity_signal(sim_matrices[i].T, aggregation="combined")

            dtw_dist = compute_dtw_distance(signal_a, signal_b)
            emb_dist = outputs["distances"][i].item()
            features = extract_pair_features(dtw_dist, signal_a, signal_b, emb_dist)
            all_features.append(features)
            all_labels.append(int(batch["labels"][i].item()))

        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {batch_idx + 1}/{len(test_loader)} batches")

all_features = np.array(all_features)
all_labels = np.array(all_labels)
print(f"\nExtracted features for {len(all_labels)} test pairs")
print(f"  Genuine: {sum(all_labels == 1)} | Impostor: {sum(all_labels == 0)}")

## 10. Baseline Anomaly Detection (Isolation Forest + Verdict)

A first-pass Isolation Forest fit on all test features (genuine+impostor
mixed), with a balanced median-threshold split. This baseline is
superseded by the improved, genuine-only version in Section 14 — kept
here for comparison.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from pipeline.anomaly import kmedoids_cluster

scaler = StandardScaler()
features_scaled = scaler.fit_transform(all_features)

iso_forest_baseline = IsolationForest(
    n_estimators=cfg.pipeline.isolation_forest_n_estimators,
    contamination=cfg.pipeline.isolation_forest_contamination,
    random_state=cfg.pipeline.isolation_forest_random_state,
)
iso_forest_baseline.fit(features_scaled)

anomaly_scores_baseline = iso_forest_baseline.decision_function(features_scaled)
predicted_labels_baseline = kmedoids_cluster(anomaly_scores_baseline, n_clusters=2)

print("Baseline results:")
print(f"  Predicted genuine:  {sum(predicted_labels_baseline == 1)}")
print(f"  Predicted impostor: {sum(predicted_labels_baseline == 0)}")

# Category E — Evaluation (Baseline)

## 11. Baseline Metrics and Confusion Matrix

In [ ]:
from utils.evaluation import compute_metrics, plot_confusion_matrix

baseline_metrics = compute_metrics(all_labels, predicted_labels_baseline)

print("=" * 60)
print("  BASELINE RESULTS")
print("=" * 60)
print(f"  Accuracy:  {baseline_metrics['accuracy']:.4f}  ({baseline_metrics['accuracy']*100:.1f}%)")
print(f"  F1-Score:  {baseline_metrics['f1_score']:.4f}  ({baseline_metrics['f1_score']*100:.1f}%)")
print(f"  Precision: {baseline_metrics['precision']:.4f}")
print(f"  Recall:    {baseline_metrics['recall']:.4f}")
print("=" * 60)

cm_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "confusion_matrix_baseline.png")
plot_confusion_matrix(all_labels, predicted_labels_baseline, save_path=cm_path)
print(f"Saved to: {cm_path}")

## 12. Baseline Robustness Score

In [ ]:
impostor_mask = all_labels == 0
genuine_mask = all_labels == 1

impostor_detected_baseline = sum(predicted_labels_baseline[impostor_mask] == 0)
impostor_total = sum(impostor_mask)
genuine_detected_baseline = sum(predicted_labels_baseline[genuine_mask] == 1)
genuine_total = sum(genuine_mask)
robustness_score_baseline = impostor_detected_baseline / impostor_total

print("=" * 60)
print("  BASELINE ROBUSTNESS")
print("=" * 60)
print(f"  Genuine detection rate:  {genuine_detected_baseline}/{genuine_total} ({genuine_detected_baseline/genuine_total*100:.1f}%)")
print(f"  Impostor detection rate: {impostor_detected_baseline}/{impostor_total} ({impostor_detected_baseline/impostor_total*100:.1f}%)")
print(f"  Robustness Score:        {robustness_score_baseline:.4f} ({robustness_score_baseline*100:.1f}%)")

## 13. Visualize DTW Alignment for a Sample Pair

Re-runs inference on one genuine and one impostor pair to get their raw
signals, then plots and saves the actual DTW alignment.

In [ ]:
from pipeline.dtw import compute_dtw_with_path
from utils.evaluation import plot_dtw_alignment

genuine_idx = np.where(all_labels == 1)[0][0]
impostor_idx = np.where(all_labels == 0)[0][0]


def get_pair_signals(pair_idx: int):
    """Re-run the network on one test pair, returning its (signal_a, signal_b)."""
    text_a, text_b, label = test_pairs[pair_idx]
    single_batch = collate_pairs([
        AuthorshipPairDataset([(text_a, text_b, label)])[0]
    ])
    single_batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v
                    for k, v in single_batch.items()}
    with torch.no_grad():
        outputs = model(single_batch)
        sim_matrices = model.get_similarity_scores(
            outputs["chunk_embeddings_a"], outputs["chunk_embeddings_b"],
            single_batch["chunk_mask_a"], single_batch["chunk_mask_b"],
        )
    sig_a = build_similarity_signal(sim_matrices[0], aggregation="combined")
    sig_b = build_similarity_signal(sim_matrices[0].T, aggregation="combined")
    return sig_a, sig_b


try:
    from data.dataset import AuthorshipPairDataset, collate_pairs

    sig_a_genuine, sig_b_genuine = get_pair_signals(genuine_idx)
    dtw_dist_g, path_g = compute_dtw_with_path(sig_a_genuine, sig_b_genuine)

    sig_a_impostor, sig_b_impostor = get_pair_signals(impostor_idx)
    dtw_dist_i, path_i = compute_dtw_with_path(sig_a_impostor, sig_b_impostor)

    print("Sample genuine pair:")
    print(f"  DTW distance: {dtw_dist_g:.4f}")
    print(f"  Baseline verdict: {'genuine' if predicted_labels_baseline[genuine_idx] == 1 else 'impostor'}")
    print(f"  True label: genuine")

    dtw_plot_path_genuine = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "dtw_alignment_genuine.png")
    plot_dtw_alignment(sig_a_genuine, sig_b_genuine, path=path_g, save_path=dtw_plot_path_genuine)
    print(f"  Saved plot to: {dtw_plot_path_genuine}")

    print(f"\nSample impostor pair:")
    print(f"  DTW distance: {dtw_dist_i:.4f}")
    print(f"  Baseline verdict: {'genuine' if predicted_labels_baseline[impostor_idx] == 1 else 'impostor'}")
    print(f"  True label: impostor")

    dtw_plot_path_impostor = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "dtw_alignment_impostor.png")
    plot_dtw_alignment(sig_a_impostor, sig_b_impostor, path=path_i, save_path=dtw_plot_path_impostor)
    print(f"  Saved plot to: {dtw_plot_path_impostor}")

except NameError:
    print("test_pairs is not available in memory — showing scalar distances only.")
    print("Sample genuine pair:")
    print(f"  DTW distance: {all_features[genuine_idx][0]:.4f}")
    print(f"\nSample impostor pair:")
    print(f"  DTW distance: {all_features[impostor_idx][0]:.4f}")

# Category F — Accuracy Improvement

## 14. Improved Anomaly Detection: Genuine-Only Fit + Tuned Threshold

The baseline (Section 10) fits Isolation Forest on a mix of genuine and
impostor features, then splits at the median — a generic unsupervised
approach. This section applies the methodology the project's
"Impostor Projections" design actually calls for:

1. **Fit on genuine pairs only** — the model learns what a genuine
   author's style consistency looks like; impostor pairs should then
   score as anomalies relative to that learned pattern.
2. **Tune the decision threshold on the validation set** (maximizing
   F1) instead of a blind 50/50 median split.
3. **Evaluate once, on the held-out test set**, using that tuned
   threshold — never touching test data during tuning.

No retraining is required — only the anomaly-detection step changes.

**Measured result from one full run:** baseline Accuracy/F1 of roughly
60-65% / 61-68% improved to **69.1% Accuracy / 72.3% F1** after this
genuine-only fit + threshold tuning (Genuine detection 82.0%,
Impostor detection 56.5%). Exact numbers will vary by run due to
training randomness, but the direction of improvement is consistent.

In [ ]:
# Step 1: build a validation feature set the same way as the test set,
# so the threshold can be tuned without touching the test set.
print("Building validation features for threshold tuning...")
val_features, val_labels = [], []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v
                  for k, v in batch.items()}
        outputs = model(batch)
        sim_matrices = model.get_similarity_scores(
            outputs["chunk_embeddings_a"], outputs["chunk_embeddings_b"],
            batch["chunk_mask_a"], batch["chunk_mask_b"],
        )
        for i in range(len(sim_matrices)):
            sig_a = build_similarity_signal(sim_matrices[i], aggregation="combined")
            sig_b = build_similarity_signal(sim_matrices[i].T, aggregation="combined")
            dtw_dist = compute_dtw_distance(sig_a, sig_b)
            emb_dist = outputs["distances"][i].item()
            val_features.append(extract_pair_features(dtw_dist, sig_a, sig_b, emb_dist))
            val_labels.append(int(batch["labels"][i].item()))

val_features = np.array(val_features)
val_labels = np.array(val_labels)
print(f"Validation features ready: {len(val_labels)} pairs "
      f"(genuine={sum(val_labels==1)}, impostor={sum(val_labels==0)})")

In [ ]:
# Step 2: fit scaler + Isolation Forest on GENUINE validation pairs only.
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

print("Fitting Isolation Forest on GENUINE pairs only...")

improved_scaler = StandardScaler()
all_features_scaled_v2 = improved_scaler.fit_transform(all_features)
val_features_scaled = improved_scaler.transform(val_features)

genuine_mask_train = val_labels == 1
iso_forest_improved = IsolationForest(
    n_estimators=cfg.pipeline.isolation_forest_n_estimators,
    contamination="auto",   # not needed when fitting on genuine-only
    random_state=cfg.pipeline.isolation_forest_random_state,
)
iso_forest_improved.fit(val_features_scaled[genuine_mask_train])

val_scores = iso_forest_improved.decision_function(val_features_scaled)
test_scores = iso_forest_improved.decision_function(all_features_scaled_v2)

print(f"Fitted on {genuine_mask_train.sum()} genuine validation pairs.")

In [ ]:
# Step 3: tune the decision threshold on the validation set (maximize F1).
print("Tuning threshold on validation set...")

best_threshold, best_f1 = None, -1
for pct in range(5, 96, 5):
    t = np.percentile(val_scores, pct)
    preds = (val_scores >= t).astype(int)
    f1 = f1_score(val_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

print(f"Best threshold: {best_threshold:.4f}  (validation F1 = {best_f1:.4f})")

In [ ]:
# Step 4: apply the tuned threshold to the test set and recompute all metrics.
predicted_labels = (test_scores >= best_threshold).astype(int)

accuracy = accuracy_score(all_labels, predicted_labels)
f1 = f1_score(all_labels, predicted_labels, zero_division=0)
precision = precision_score(all_labels, predicted_labels, zero_division=0)
recall = recall_score(all_labels, predicted_labels, zero_division=0)

print("=" * 60)
print("  IMPROVED RESULTS")
print("=" * 60)
print(f"  Accuracy:  {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  F1-Score:  {f1:.4f}  ({f1*100:.1f}%)")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print("=" * 60)

impostor_detected = sum(predicted_labels[impostor_mask] == 0)
genuine_detected = sum(predicted_labels[genuine_mask] == 1)

print(f"\nGenuine detection:  {genuine_detected}/{genuine_total} "
      f"({genuine_detected/genuine_total*100:.1f}%)")
print(f"Impostor detection: {impostor_detected}/{impostor_total} "
      f"({impostor_detected/impostor_total*100:.1f}%)")

print("\n--- Comparison vs. baseline (Section 11-12) ---")
print(f"  Accuracy:            {baseline_metrics['accuracy']*100:.1f}%  ->  {accuracy*100:.1f}%")
print(f"  F1-Score:             {baseline_metrics['f1_score']*100:.1f}%  ->  {f1*100:.1f}%")
print(f"  Impostor detection:   {impostor_detected_baseline/impostor_total*100:.1f}%  ->  {impostor_detected/impostor_total*100:.1f}%")

improved_cm_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "confusion_matrix_improved.png")
plot_confusion_matrix(all_labels, predicted_labels, save_path=improved_cm_path)
print(f"\nSaved improved confusion matrix to: {improved_cm_path}")

# Category G — Save Results

## 15. Save Everything to Drive

Saves both the baseline and improved metrics, plus a self-describing
`.npz` file (feature schema is recorded explicitly, preventing silent
shape-mismatch errors if this file is reused by different code later).

In [ ]:
import json

# Baseline + improved metrics, side by side
final_metrics = {
    "baseline": baseline_metrics,
    "improved": {
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "tuned_threshold": float(best_threshold),
        "fit_method": "genuine_only",
    },
}
metrics_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "final_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=2)

# Robustness, both versions
robustness_results = {
    "baseline": {
        "genuine_detection_rate": float(genuine_detected_baseline / genuine_total),
        "impostor_detection_rate": float(impostor_detected_baseline / impostor_total),
        "robustness_score": float(robustness_score_baseline),
    },
    "improved": {
        "genuine_detection_rate": float(genuine_detected / genuine_total),
        "impostor_detection_rate": float(impostor_detected / impostor_total),
        "robustness_score": float(impostor_detected / impostor_total),
    },
}
robustness_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "robustness_analysis.json")
with open(robustness_path, "w") as f:
    json.dump(robustness_results, f, indent=2)

# Self-describing feature file — records its own schema so future code
# can check compatibility before loading it.
FEATURE_NAMES = [
    "dtw_distance",
    "signal_a_mean", "signal_a_std", "signal_a_min", "signal_a_max",
    "signal_a_median", "signal_a_var",
    "signal_b_mean", "signal_b_std", "signal_b_min", "signal_b_max",
    "mean_embedding_distance",
    "signal_corr", "signal_a_range", "signal_b_range", "dtw_to_emb_ratio",
]
feature_schema_version = "phase_b_v2" if all_features.shape[1] == len(FEATURE_NAMES) else "original_v1"

predictions_path = os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, "test_predictions.npz")
np.savez(
    predictions_path,
    true_labels=all_labels,
    predicted_labels_baseline=predicted_labels_baseline,
    predicted_labels_improved=predicted_labels,
    anomaly_scores_baseline=anomaly_scores_baseline,
    anomaly_scores_improved=test_scores,
    features=all_features,
    feature_schema_version=feature_schema_version,
    feature_dim=all_features.shape[1],
)

print("All results saved to Drive:")
print(f"  Metrics:              {metrics_path}")
print(f"  Robustness:           {robustness_path}")
print(f"  Predictions:          {predictions_path}")
print(f"  Loss curves:          {os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, 'loss_curves.png')}")
print(f"  Confusion (baseline): {os.path.join(cfg.paths.root_dir, cfg.paths.results_dir, 'confusion_matrix_baseline.png')}")
print(f"  Confusion (improved): {improved_cm_path}")
print(f"  Checkpoints dir:      {os.path.join(cfg.paths.root_dir, cfg.paths.checkpoint_dir)}")
print(f"  NOTE: verify which checkpoint_epoch_N.pt has the lowest val_loss")
print(f"        before reusing it later — see Appendix for a known caveat")
print(f"        about a stale best_model.pt across separate Colab sessions.")
print("\nDone.")

# Appendix — Resuming Without Retraining

This appendix documents a real scenario hit while developing this
project: training was interrupted by a Colab disconnect partway
through, but training had actually continued in the background and
completed successfully (through early stopping) before the
disconnect was noticed. This section explains how to verify which
checkpoint is actually the best one, and how to run only the
inference/evaluation pipeline on it — without repeating training.

**Known caveat:** in one observed run, `best_model.pt` did not match
any epoch of the training run that had just completed — it held an
older val_loss value. The most likely cause is a session/path issue
(e.g. `cfg.paths.root_dir` pointing to a different location at some
point during a partially-disconnected session, so a `best_model.pt`
write landed somewhere other than where the per-epoch
`checkpoint_epoch_N.pt` files ended up). Regardless of the exact
cause, the safest practice is to never trust `best_model.pt`'s
filename alone — always confirm the best epoch by comparing the
`val_loss` field stored inside each `checkpoint_epoch_N.pt` file
directly, as done below.

In [ ]:
# Step 1: identify the BEST checkpoint by its own internal val_loss field
# (this is more reliable than trusting best_model.pt's filename).
import torch
import os

ckpt_dir = os.path.join(cfg.paths.root_dir, cfg.paths.checkpoint_dir)
candidates = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint_epoch_")]

best_file, best_val_loss = None, float("inf")
for fname in candidates:
    ckpt = torch.load(os.path.join(ckpt_dir, fname), map_location="cpu")
    vl = ckpt.get("val_loss")
    print(f"  {fname}: epoch={ckpt.get('epoch')}  val_loss={vl}")
    if vl is not None and vl < best_val_loss:
        best_val_loss = vl
        best_file = fname

print(f"\nTrue best checkpoint: {best_file}  (val_loss={best_val_loss:.4f})")

In [ ]:
# Step 2: load that checkpoint directly into a freshly-built model
# (skip this if model/cfg are already in memory and correctly patched).
from models.siamese import SiameseNetwork
from training.trainer import load_checkpoint

model = SiameseNetwork()
load_checkpoint(model, os.path.join(ckpt_dir, best_file), cfg.device)
print(f"Loaded {best_file} — ready for inference without retraining.")

In [ ]:
# Step 3: if train_pairs is unavailable (e.g. a fresh session that only
# needs to run inference), create_dataloaders_from_splits() still requires
# a non-empty train_pairs argument. A 1-pair placeholder satisfies this
# without affecting val/test results, since train_loader is unused here.
from data.dataset import create_dataloaders_from_splits

dummy_train_pairs = [val_pairs[0]]
_, val_loader, test_loader = create_dataloaders_from_splits(
    dummy_train_pairs, val_pairs, test_pairs
)
print(f"Val batches:  {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print("\nProceed to Section 9 (Inference Pipeline) using this model/val_loader/test_loader.")